# OpenAI Integration with MCP - Step-by-Step

This notebook breaks down the `client-simple.py` script into executable cells. It connects to the `server.py` MCP server, gets its available tools, and uses an OpenAI model (`gpt-4o`) to answer a question by allowing the model to decide when to call the MCP tool.

We'll add plenty of `print()` statements so you can see exactly what happens under the hood.

## 1. Imports and Setup

First, we import the necessary libraries, load our environment variables (importantly, `OPENAI_API_KEY`), and set up our global variables to track the state of our session.

In [7]:
import asyncio
import json
from contextlib import AsyncExitStack
from typing import Any, Dict, List

import nest_asyncio
from dotenv import load_dotenv
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from openai import AsyncOpenAI
import subprocess

# Apply nest_asyncio to allow nested event loops in Jupyter
nest_asyncio.apply()

# Load environment variables (Make sure your .env has OPENAI_API_KEY)
load_dotenv("../.env", override= True)
print("Environment variables loaded.")

# Global variables to store session state
session = None
exit_stack = AsyncExitStack()
openai_client = AsyncOpenAI()
model = "gpt-4o-mini"
stdio = None
write = None
print("Global variables initialized.")

Environment variables loaded.
Global variables initialized.


## 2. Connect to the MCP Server

We'll define a function to start our `server.py` and connect to it using the `stdio` transport. We immediately run it to establish the connection.

In [8]:
async def connect_to_server(server_script_path: str = "server.py"):
    """Connect to an MCP server."""
    global session, stdio, write, exit_stack
    print(f"Connecting to MCP server using script: {server_script_path}...")
    
    # Server configuration
    server_params = StdioServerParameters(
        command="python",
        args=[server_script_path],
    )

    # Connect to the server
    stdio_transport = await exit_stack.enter_async_context(stdio_client(server_params, errlog=subprocess.DEVNULL))
    stdio, write = stdio_transport
    session = await exit_stack.enter_async_context(ClientSession(stdio, write))

    # Initialize the connection
    await session.initialize()
    print("Server connection initialized!")

    # List available tools
    tools_result = await session.list_tools()
    print("\nConnected to server with tools:")
    for tool in tools_result.tools:
        print(f"  - {tool.name}: {tool.description}")

# Establish the connection now
await connect_to_server("server.py")

Connecting to MCP server using script: server.py...
Server connection initialized!

Connected to server with tools:
  - get_knowledge_base: Retrieve the entire knowledge base as a formatted string.

    Returns:
        A formatted string containing all Q&A pairs from the knowledge base.
    


## 3. Formatting Tools for OpenAI

OpenAI requires tool descriptions in a specific JSON schema. This function retrieves the tools from the MCP server and reformats them so OpenAI can understand them.

In [9]:
async def get_mcp_tools() -> List[Dict[str, Any]]:
    """Get available tools from the MCP server in OpenAI format."""
    global session
    print("Fetching tools from MCP server to pass to OpenAI...")
    
    tools_result = await session.list_tools()
    tools_list = [
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description,
                "parameters": tool.inputSchema,
            },
        }
        for tool in tools_result.tools
    ]
    
    print(f"Formatted {len(tools_list)} tool(s) for OpenAI format.")
    return tools_list

## 4. Processing the Query

This is the core logic. It sends a user query to OpenAI along with the available tools. If OpenAI decides it needs to call a tool, it will execute the tool via the MCP session, and then send the result back to OpenAI to get the final natural language answer.

In [10]:
async def process_query(query: str) -> str:
    """Process a query using OpenAI and available MCP tools."""
    global session, openai_client, model
    print(f"\n--- Processing query: '{query}' ---")

    # Get available tools formatted for OpenAI
    tools = await get_mcp_tools()

    # 1. Initial OpenAI API call
    print("\n1. Sending initial request to OpenAI with tool list...")
    response = await openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": query}],
        tools=tools,
        tool_choice="auto",
    )

    # Get assistant's response
    assistant_message = response.choices[0].message

    # Initialize conversation history with user query and assistant response
    messages = [
        {"role": "user", "content": query},
        assistant_message,
    ]

    # Handle tool calls if present
    if assistant_message.tool_calls:
        print(f"\n2. OpenAI decided to call {len(assistant_message.tool_calls)} tool(s)!")
        
        # Process each tool call
        for tool_call in assistant_message.tool_calls:
            print(f"  -> Executing tool: {tool_call.function.name}")
            print(f"  -> With arguments: {tool_call.function.arguments}")
            
            # Execute tool call on the MCP Server
            result = await session.call_tool(
                tool_call.function.name,
                arguments=json.loads(tool_call.function.arguments),
            )
            
            print(f"  -> Tool execution complete. Result returned {len(result.content[0].text)} characters.")

            # Add tool response to conversation history
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result.content[0].text,
                }
            )

        # 3. Get final response from OpenAI with tool results
        print("\n3. Sending tool results back to OpenAI for a final answer...")
        final_response = await openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="none",  # Don't allow more tool calls
        )
        print("  -> Received final answer from OpenAI.")
        return final_response.choices[0].message.content

    # No tool calls, just return the direct response
    print("\n2. No tool calls were needed. Returning direct response.")
    return assistant_message.content

## 5. Execution Time!

Let's ask a question that requires the `get_knowledge_base` tool to answer.

In [11]:
query = "What is our company's vacation policy?"

# This will trigger the whole pipeline
response = await process_query(query)

print("\n" + "="*50)
print("FINAL RESPONSE:\n")
print(response)
print("="*50)


--- Processing query: 'What is our company's vacation policy?' ---
Fetching tools from MCP server to pass to OpenAI...
Formatted 1 tool(s) for OpenAI format.

1. Sending initial request to OpenAI with tool list...

2. OpenAI decided to call 1 tool(s)!
  -> Executing tool: get_knowledge_base
  -> With arguments: {}
  -> Tool execution complete. Result returned 1987 characters.

3. Sending tool results back to OpenAI for a final answer...
  -> Received final answer from OpenAI.

FINAL RESPONSE:

Our company's vacation policy states that full-time employees are entitled to 20 paid vacation days per year. Here are the key details:

- **Eligibility:** Vacation days can be taken after completing 6 months of employment.
- **Carryover:** Unused vacation days can be carried over to the next year, up to a maximum of 5 days.
- **Requesting Time Off:** Vacation requests should be submitted at least 2 weeks in advance through the HR portal.
